In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
from pathlib import Path
BASE_PATH = Path(os.getcwd()).parent
DATA_PATH = (BASE_PATH / "data" / "processed")


In [3]:
train = pd.read_csv(DATA_PATH / "train.csv")

In [4]:
train = train.drop(train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 12.5)].index)

In [5]:
X = train.drop(['SalePrice'],axis=1)
y = train['SalePrice']

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

models = {
    'Random Forest': RandomForestRegressor(),
    'GradientBoost': GradientBoostingRegressor(),
    'Ridge': Ridge(),
    'XGBoost': XGBRegressor(),
    'CatBoost': CatBoostRegressor(verbose=False)
}

In [7]:
from sklearn.model_selection import cross_val_score

for model in models:
    scores = cross_val_score(
        estimator=models[model],
        X=X,
        y=y,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )
    
    rmse = -scores  # convert back to positive RMSE
    print(model, "Mean RMSE:", rmse.mean(), "Std:", rmse.std(), "Var:", rmse.var())

Random Forest Mean RMSE: 0.1393256954656229 Std: 0.006711618769438762 Var: 4.504582650628268e-05
GradientBoost Mean RMSE: 0.12375302260739714 Std: 0.011762994246975716 Var: 0.0001383680336543838
Ridge Mean RMSE: 0.132039627555205 Std: 0.0167850091798376 Var: 0.0002817365331672325
XGBoost Mean RMSE: 0.14057333200466537 Std: 0.010274508323002671 Var: 0.00010556552127945116
CatBoost Mean RMSE: 0.11833866961398401 Std: 0.012269997186345685 Var: 0.000150552830952931


In [8]:
from sklearn.model_selection import RandomizedSearchCV

cat_params = {
    "iterations": [1000, 2000, 3000],
    "learning_rate": [0.01, 0.03, 0.05],
    "depth": [4, 6, 8, 10],
    "l2_leaf_reg": [1, 3, 5, 7, 9],
    "bagging_temperature": [0, 0.5, 1],
    "random_strength": [1, 5, 10],
    "border_count": [64, 128, 254]
}

cat_cv = RandomizedSearchCV(
    CatBoostRegressor(verbose=False, loss_function='RMSE',random_state=42),
    cat_params,
    cv=3,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

cat_cv.fit(X, y)

print(f'CatBoost CV score: {cat_cv.best_score_:.4f}')
print('Params:', cat_cv.best_params_)

res = cross_val_score(
    cat_cv.best_estimator_,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

print("rmse cv_score:", -res.mean())
print("std cv_score:", res.std())

CatBoost CV score: -0.1203
Params: {'random_strength': 5, 'learning_rate': 0.01, 'l2_leaf_reg': 3, 'iterations': 3000, 'depth': 6, 'border_count': 64, 'bagging_temperature': 1}
rmse cv_score: 0.11764295808012215
std cv_score: 0.012788981526034339


In [9]:
gbr_params = {
    "n_estimators": [500, 1000, 2000],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "subsample": [0.6, 0.8, 1.0],
    "max_features": ["sqrt", "log2", None]
}

gbr_cv = RandomizedSearchCV(
    GradientBoostingRegressor(),
    gbr_params,
    cv=3,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

gbr_cv.fit(X,y)

print(f'GradBoost CV score: {gbr_cv.best_score_:.4f}')
print('Params:', gbr_cv.best_params_)

res = cross_val_score(
    gbr_cv.best_estimator_,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

print("rmse cv_score:", -res.mean())
print("std cv_score:", res.std())

GradBoost CV score: -0.1233
Params: {'subsample': 0.8, 'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 2, 'learning_rate': 0.03}
rmse cv_score: 0.12224921326059786
std cv_score: 0.012142118429517552


In [10]:
ridge_params = {
    "alpha": [0.01, 0.1, 1, 5, 10, 20, 50, 100],
    "solver": ["auto", "svd", "cholesky", "lsqr", "saga"],
    "fit_intercept": [True, False]
}

ridge_cv = RandomizedSearchCV(
    Ridge(),
    ridge_params,
    cv=3,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

ridge_cv.fit(X,y)

print(f'Ridge CV score: {ridge_cv.best_score_:.4f}')
print('Params:', ridge_cv.best_params_)

res = cross_val_score(
    ridge_cv.best_estimator_,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

print("rmse cv_score:", -res.mean())
print("std cv_score:", res.std())

Ridge CV score: -0.1272
Params: {'solver': 'auto', 'fit_intercept': True, 'alpha': 10}
rmse cv_score: 0.1258459768613306
std cv_score: 0.01619871172925653


In [11]:
cat = cat_cv.best_estimator_
gbr = gbr_cv.best_estimator_
ridge = ridge_cv.best_estimator_

In [12]:
X_test = pd.read_csv(DATA_PATH / "test.csv")

In [13]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=7, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
oof_gbr = np.zeros(len(X))
oof_ridge = np.zeros(len(X))

test_cat = np.zeros(len(X_test))
test_gbr = np.zeros(len(X_test))
test_ridge = np.zeros(len(X_test))

In [14]:
for train_idx, val_idx in kf.split(X):

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    cat.fit(X_tr, y_tr)
    gbr.fit(X_tr, y_tr)
    ridge.fit(X_tr, y_tr)

    oof_cat[val_idx] = cat.predict(X_val)
    oof_gbr[val_idx] = gbr.predict(X_val)
    oof_ridge[val_idx] = ridge.predict(X_val)

    test_cat += cat.predict(X_test) / kf.n_splits
    test_gbr += gbr.predict(X_test) / kf.n_splits
    test_ridge += ridge.predict(X_test) / kf.n_splits

In [15]:
stack_train = np.column_stack((oof_cat, oof_gbr, oof_ridge))
stack_test = np.column_stack((test_cat, test_gbr, test_ridge))

In [16]:
meta = Ridge(alpha=10)
meta.fit(stack_train, y)

final_pred = meta.predict(stack_test)

In [17]:
test_df = pd.read_csv(DATA_PATH / "test.csv")

In [20]:
os.makedirs(BASE_PATH / "submission", exist_ok=True)


submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": np.expm1(final_pred)
})

submission.to_csv(BASE_PATH / "submission/OOF_Meta.csv", index=False)

# Save Tuned CatBoost Results
cat_pred = cat_cv.best_estimator_.predict(test_df)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": np.expm1(cat_pred)
})

submission.to_csv(BASE_PATH / "submission/CatBoost.csv", index=False)

# Save Tuned Gradient Boosting Results
gbr_pred = gbr_cv.best_estimator_.predict(test_df)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": np.expm1(gbr_pred)
})

submission.to_csv(BASE_PATH / "submission/GradBoost.csv", index=False)

# Save Tuned Ridge Results
ridge_pred = ridge_cv.best_estimator_.predict(test_df)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": np.expm1(ridge_pred)
})

submission.to_csv(BASE_PATH / "submission/Ridge.csv", index=False)